In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/lopezjatjat10@gmail.com/sportsbar-etl-proj/1_setup/utilities

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "gross_price", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
df_bronze = spark.sql(f'SELECT * FROM {catalog}.{bronze_schema}.{data_source}')

display(df_bronze)

In [0]:
##normalize month
## handle gross price
## Get the price for each product_code (aggregated by year)

In [0]:
df_bronze.select('month').distinct().show()

In [0]:

date_formats = ["yyyy/MM/dd", "dd/MM/yyyy", "yyyy-MM-dd", "dd-MM-yyyy"]

df_silver = df_bronze.withColumn(
    "month",
    F.coalesce(
        F.try_to_date(F.col("month"), "yyyy/MM/dd"),
        F.try_to_date(F.col("month"), "dd/MM/yyyy"),
        F.try_to_date(F.col("month"), "yyyy-MM-dd"),
        F.try_to_date(F.col("month"), "dd-MM-yyyy")
    )
)

In [0]:
df_silver.select('month').distinct().show()

In [0]:
display(df_silver)

In [0]:
df_silver = df_silver.withColumn(
    'gross_price',
    F.when(F.col("gross_price").rlike(r'^-?\d+(\.\d+)?$'),
           F.when(F.col("gross_price").cast('double')  < 0, -1 * F.col('gross_price').cast('double'))
           .otherwise(F.col("gross_price").cast('double')))
)


df_silver.show(10)


In [0]:
df_products = spark.table('fmcg.silver.products')
display(df_products.limit(10))

In [0]:
df_joined = df_silver.join(df_products.select('product_id', 'product_code'), on='product_id', how='inner')
df_joined = df_joined.select('product_id','product_code', 'month', 'gross_price', 'read_timestamp', 'file_name', 'file_size')

display(df_joined.limit(10))


In [0]:
df_joined.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed', 'True')\
    .option('mergeSchema', 'True')\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')